# 第19章　評価指標をコードで実装する

**『医療診断支援AI開発　基礎編 ― 自分で作る（基礎編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-basic

## 19.1　分類 ― 混同行列から

In [ ]:
from sklearn.metrics import (confusion_matrix, recall_score, precision_score,
    f1_score, roc_auc_score, average_precision_score)

# y_true: 正解(0/1), y_pred: 予測(0/1), y_prob: 陽性確率
tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
sensitivity = tp / (tp + fn)                     # 感度
specificity = tn / (tn + fp)                     # 特異度
precision   = precision_score(y_true, y_pred)    # 適合率
f1          = f1_score(y_true, y_pred)           # F1
roc_auc     = roc_auc_score(y_true, y_prob)      # ROC-AUC
ap          = average_precision_score(y_true, y_prob)  # AP（average precision。不均衡で重要）
print(f"感度{sensitivity:.3f} 特異度{specificity:.3f} AP{ap:.3f}")

## 出力を読む ― classification_report の worked example

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_true, y_pred, target_names=["正常", "病変"], digits=3))

## 19.2　信頼区間を添える

In [ ]:
import numpy as np

def bootstrap_ci(y_true, y_pred, metric, n=1000, valid=None):
    """1行が1人の独立した患者である場合の例。
    validは、その再標本で指標が定義できるかを判定する関数。
    複数画像が同一患者に属する場合は、社会実装編の統計の章の群単位の抽出を使う。
    """
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    if len(y_true) == 0 or len(y_true) != len(y_pred):
        return None, 0
    if valid is None:
        valid = lambda y: len(np.unique(y)) >= 2   # 既定はROC-AUCなど両クラスが必要な指標向け
    vals = []
    rng = np.random.default_rng(0)
    for _ in range(n):
        idx = rng.integers(0, len(y_true), len(y_true))
        if not valid(y_true[idx]):
            continue
        value = metric(y_true[idx], y_pred[idx])
        if np.isfinite(value):
            vals.append(value)
    if not vals:
        return None, 0
    return np.percentile(vals, [2.5, 97.5]), len(vals)

ci, n_valid = bootstrap_ci(y_true, y_pred, recall_score,
                           valid=lambda y: np.any(y == 1))
if ci is None:
    print("感度の信頼区間は評価不能（有効な再標本がない）")
else:
    lo, hi = ci
    print(f"感度の95%信頼区間: {lo:.3f}〜{hi:.3f}（有効反復数 {n_valid}）")

## 19.3　セグメンテーション ― DiceとIoU

In [ ]:
def dice(pred, gt, eps=1e-6):
    pred, gt = pred.astype(bool), gt.astype(bool)
    inter = (pred & gt).sum()
    return (2*inter + eps) / (pred.sum() + gt.sum() + eps)

def iou(pred, gt, eps=1e-6):
    pred, gt = pred.astype(bool), gt.astype(bool)
    inter = (pred & gt).sum(); union = (pred | gt).sum()
    return (inter + eps) / (union + eps)

## 19.4　検出 ― 病変単位のマッチング

```text
!pip install connected-components-3d      # import名は cc3d（配布名と異なる）
import cc3d
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_bipartite_matching

def lesion_match(pred_mask, gt_mask, overlap_thr=0.1):   # 重なり=GT被覆率（IoUではない）
    pc = cc3d.connected_components(pred_mask)
    gc = cc3d.connected_components(gt_mask)
    n_pred, n_gt = int(pc.max()), int(gc.max())
    # ① 候補の辺を張る：正解病変 g を予測塊 p が被覆率 overlap_thr 以上で覆っていれば候補
    #    （被覆率 = 交わり ÷ 正解病変のボクセル数）
    rows, cols = [], []
    for g in range(1, n_gt+1):
        gt = (gc == g); gt_n = float(gt.sum())
        for p in range(1, n_pred+1):
            if float(((pc == p) & gt).sum()) / gt_n >= overlap_thr:
                rows.append(g-1); cols.append(p-1)
    # ② 1対1の対応数を最大にする（二部グラフの最大マッチング）。
    #    正解を番号順にたどって「最もよく覆う予測を確保する」貪欲法だと、先に処理した正解が
    #    別の正解の唯一の候補を奪い、同じ配置を左右反転しただけでTPが変わる。
    #    最大マッチングなら、対応の組み合わせが複数あっても対応数は走査順に依存しない。
    tp = 0
    if rows:
        adj = csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(n_gt, n_pred))
        match = maximum_bipartite_matching(adj, perm_type="column")   # 正解ごとに対応する予測（無ければ -1）
        tp = int((match >= 0).sum())
    fp = n_pred - tp                            # どの正解にも対応づかなかった予測塊
    # 病変が1個も無い症例の検出率は「0」ではなく「測れない」（本章末の規約と同じ）
    return {"tp": tp, "fp": fp, "n_gt": n_gt, "n_pred": n_pred,
            "lesion_recall":    tp / n_gt   if n_gt   else float("nan"),
            "lesion_precision": tp / n_pred if n_pred else float("nan"),
            # 塗りすぎを見抜くため、予測と正解の総体積を必ず持ち帰る
            "pred_voxels": int(pred_mask.sum()), "gt_voxels": int(gt_mask.sum())}
```

## 実装した指標を検算する ― 「答えの分かっている入力」で確かめる

In [ ]:
import numpy as np
def dice(pred, gt, eps=1e-6):
    pred, gt = pred.astype(bool), gt.astype(bool)
    inter = (pred & gt).sum()
    return (2*inter + eps) / (pred.sum() + gt.sum() + eps)

a = np.array([1, 1, 0, 0])
assert abs(dice(a, a)   - 1.0) < 1e-3        # 完全一致 → 1
assert dice(a, 1 - a) < 1e-3                  # まったく重ならない → 0
p = np.array([1, 1, 0, 0]); g = np.array([0, 1, 1, 0])   # 重なり1画素、面積2+2
assert abs(dice(p, g) - 0.5) < 1e-3           # 2*1/(2+2) = 0.5
print("dice OK")

In [ ]:
from sklearn.metrics import confusion_matrix
y_true = [0, 0, 0]; y_pred = [0, 0, 0]        # 陽性ゼロの検証バッチ（小バッチで起こる）
# confusion_matrix(y_true, y_pred).ravel()  → 要素1個で ValueError
tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()  # labelsで常に2x2
sens = tp / (tp + fn) if (tp + fn) else float("nan")   # 分母0はnanに（0除算を避ける）
print(tn, fp, fn, tp, "sensitivity =", sens)